# 06g — Structured Policy-object Extraction and Audit

**Notebook version: 06g-policy-object-v6**

本 Notebook 从 motion title、motion text、motion type 和 legislation name 中提取结构化政策对象。它不会调用 API 或 LLM，也不需要 API Key。

核心原则：

1. 区分本次表决命题（`vote_proposition`）和底层政策对象（`canonical_policy_object`）；
2. 单独记录 Aye 对政策对象的作用方向（`aye_effect_on_object`）；
3. 每个自动结果必须保留来自原始文本的 `source_quote`；
4. 优先使用已有人工作业，其次使用按 motion type 编写的规则，最后才允许标题兜底；
5. 多议题、对象过于宽泛、规则冲突或低置信度记录进入人工队列，而不是强行预测；
6. 本步骤只改善查询定义，不读取政党投票结果，也不运行最终 Test。

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 所有路径都集中在这里，方便迁移项目。
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'processed'
DIVISION_PATH = PROCESSED_DIR / 'model_v2' / 'division_labels_final.csv'
MODEL_READY_PATH = PROCESSED_DIR / 'model_v2' / 'model_ready_v2_all.csv'
V4_OVERRIDE_PATH = PROCESSED_DIR / 'rag_audit_v4' / 'query_definition_overrides_v4.csv'
V5_EVIDENCE_PATH = PROCESSED_DIR / 'rag_audit_v5' / 'evidence_role_audit_v5.csv'
OUTPUT_DIR = PROCESSED_DIR / 'policy_object_v6'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_VERSION = '06g-policy-object-v6'
MAX_OBJECT_CHARS = 520
MAX_QUOTE_CHARS = 700
GENERIC_OBJECTS = {
    'social security', 'finance bill', 'other', 'business of the house',
    'government agenda', 'immigration and home affairs',
    'health', 'education', 'economy', 'transport',
}

print('Notebook version:', NOTEBOOK_VERSION)
print('API calls: 0')
print('LLM calls: 0')

Notebook version: 06g-policy-object-v6
API calls: 0
LLM calls: 0


## 1. Load one row per division

`division_labels_final.csv` 提供 motion 原文和已有人工作业；`model_ready_v2_all.csv` 只补充 motion family、政策领域和文本长度等预测前字段。

In [2]:
# 读取 division 级数据，不使用政党投票标签进行提取。
divisions = pd.read_csv(DIVISION_PATH, low_memory=False)
model_ready = pd.read_csv(MODEL_READY_PATH, low_memory=False)

meta_columns = [
    'division_key', 'motion_family', 'policy_domain_primary',
    'policy_domain_count', 'motion_char_count',
]
division_meta = (
    model_ready[meta_columns]
    .drop_duplicates('division_key')
)

divisions = divisions.merge(
    division_meta, on='division_key', how='left', validate='one_to_one'
)
divisions['motion_date'] = pd.to_datetime(divisions['motion_date'], errors='coerce')

if V4_OVERRIDE_PATH.exists():
    overrides = pd.read_csv(V4_OVERRIDE_PATH)
else:
    overrides = pd.DataFrame(columns=[
        'division_key', 'override_action', 'policy_object_override',
        'override_reason',
    ])

divisions = divisions.merge(
    overrides, on='division_key', how='left', validate='one_to_one'
)

print('Divisions:', len(divisions))
print('Existing manually reviewed policy objects:', divisions['final_policy_object'].notna().sum())
print('V4 query overrides:', len(overrides))
display(divisions.head(3))

Divisions: 2091
Existing manually reviewed policy objects: 263
V4 query overrides: 5


,division_key,motion_date,split,motion_type,motion_title_clean,motion_text_clean,legislation_name_clean,final_motion_polarity,final_polarity_source,final_polarity_confidence,...,final_object_government_backed,final_government_backing_source,government_backing_known,motion_family,policy_domain_primary,policy_domain_count,motion_char_count,override_action,policy_object_override,override_reason
0,pw-2016-01-06-157-commons,2016-01-06,train,other,Opposition Day — [14th Allotted Day] — Univers...,"I beg to move, That this House calls on the Go...",NaN,-1.0,rule_high,high,...,NaN,unknown,0,other_motion,welfare_pensions_poverty,1.0,170.0,NaN,NaN,NaN
1,pw-2016-01-06-158-commons,2016-01-06,train,other,Opposition Day — [14th Allotted Day] — Univers...,"Question put, The House divided: Ayes 216, Noe...",NaN,NaN,excluded,not_usable,...,NaN,unknown,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pw-2016-01-07-159-commons,2016-01-07,train,other,State Pension Age (Women),"Resolved, That this House, while welcoming the...",NaN,NaN,excluded,not_usable,...,NaN,unknown,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Text and evidence-span helpers

自动提取的结论必须带有原文依据。`source_quote_anchored=True` 表示引用确实可以在标准化后的 title、motion text 或 legislation name 中找到。

In [3]:
def safe_text(value):
    # 将缺失值转换为空字符串并统一空白字符。
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def clean_stage_prefix(text):
    # 删除标题中的程序阶段前缀，但保留 Bill 或政策实体名称。
    value = safe_text(text)
    patterns = [
        r'^(?:Approve|Revoke|Annul):\s*',
        r'^(?:Second|Third) Reading:\s*',
        r'^Bill Introduction:\s*',
        r'^Leave for Bill:\s*',
        r'^New Clause\s+\d+[:—-]\s*',
        r'^Clause\s+\d+[:—-]\s*',
        r'^Commencement:\s*\d*\.?\s*',
        r'^Income Tax \(Charge\):\s*\d*\.?\s*',
    ]
    for pattern in patterns:
        value = re.sub(pattern, '', value, flags=re.I)
    return value.strip(' —:-')


def content_tokens(text):
    # 词元只用于寻找最能支持人工对象的原文句子，不用于生成标签。
    stop = {
        'that', 'this', 'with', 'from', 'into', 'have', 'been', 'will',
        'would', 'should', 'could', 'house', 'motion', 'bill', 'clause',
        'government', 'policy', 'proposed', 'amendment', 'reading',
    }
    return {
        token for token in re.findall(r"[a-z][a-z'-]{2,}", safe_text(text).lower())
        if token not in stop
    }


def best_source_quote(object_text, title, motion_text, legislation=''):
    # 从原始字段中选择与政策对象重合最多的原文片段。
    target = content_tokens(object_text)
    sources = [safe_text(motion_text), safe_text(title), safe_text(legislation)]
    candidates = []
    for source_rank, source in enumerate(sources):
        if not source:
            continue
        pieces = re.split(r'(?<=[.;!?])\s+|\s+(?=That this House\b)', source)
        for piece in pieces:
            piece = safe_text(piece)
            if len(piece) < 12:
                continue
            overlap = len(target & content_tokens(piece))
            coverage = overlap / max(len(target), 1)
            candidates.append((coverage, overlap, -source_rank, -len(piece), piece))
    if not candidates:
        return ''
    quote = max(candidates)[-1]
    return quote[:MAX_QUOTE_CHARS].strip()


def quote_is_anchored(quote, title, motion_text, legislation=''):
    # 检查引用是否来自输入文本，避免生成不存在的依据。
    quote = safe_text(quote)
    haystack = ' || '.join([
        safe_text(title), safe_text(motion_text), safe_text(legislation)
    ])
    return bool(quote) and quote in haystack

## 3. Motion-type-aware extraction rules

规则优先识别具有明确法律效果的表达，例如 `be annulled`、`leave out clause`、`at end insert`、`bring in a Bill to`。只有规则和已有人工对象都不可用时才回退到标题。

In [4]:
INSTRUMENT_PATTERN = re.compile(
    r'(?P<instrument>(?:the\s+)?(?:[A-Za-z0-9’\'()&/–-]+\s+){1,18}'
    r'(?:Regulations|Order|Rules)\s+20\d{2}(?:\s*\([^)]*\))?)'
    r'(?P<tail>.{0,90}?)(?P<action>be\s+annulled|be\s+revoked|be\s+approved)',
    flags=re.I,
)

NEW_CLAUSE_PATTERN = re.compile(
    r'(?:New Clause\s+\d+\s*[-—:]\s*)(?P<heading>[^.]{8,220}?)(?=\s*\(1\)|\s+The\s|\.|$)',
    flags=re.I,
)

BRING_BILL_PATTERN = re.compile(
    r'(?:leave be given to bring in a Bill to|bring in a Bill to)\s+(?P<object>[^.]{12,420})',
    flags=re.I,
)

CALLS_ON_PATTERN = re.compile(
    r'(?:calls? on|urges?)\s+(?:the\s+)?(?:Government|Secretary of State|Ministers?)\s+to\s+'
    r'(?P<object>[^.]{12,420})',
    flags=re.I,
)

RESOLVED_PATTERN = re.compile(
    r'(?:That this House|Resolved,? That this House)\s+(?P<object>[^.]{15,500})',
    flags=re.I,
)

ACTION_RULES = [
    ('annul', re.compile(r'\bannul(?:led|ment)?\b', re.I), -1),
    ('revoke', re.compile(r'\brevok(?:e|ed|ing|ation)\b', re.I), -1),
    ('remove_or_delete', re.compile(r'\b(?:leave out|remove|delete|repeal)\b', re.I), -1),
    ('abolish', re.compile(r'\b(?:abolish|abolition|scrap|end|reverse)\b', re.I), -1),
    ('replace', re.compile(r'\bleave out\b.{0,120}\binsert\b', re.I), 0),
    ('insert_or_add', re.compile(r'\b(?:at end insert|insert|add|new clause)\b', re.I), 1),
    ('approve', re.compile(r'\b(?:approve|approved)\b', re.I), 1),
    ('introduce', re.compile(r'\b(?:introduce|bring in)\b', re.I), 1),
    ('increase', re.compile(r'\b(?:increase|raise|extend|expand)\b', re.I), 1),
    ('reduce', re.compile(r'\b(?:reduce|cut|restrict|limit)\b', re.I), -1),
    ('require', re.compile(r'\b(?:require|must|place a duty)\b', re.I), 1),
    ('review', re.compile(r'\b(?:review|assessment|report)\b', re.I), 1),
]


def infer_action(text, motion_type=''):
    # 先识别复合动作和替换，因为它们包含不止一个方向。
    text = safe_text(text)
    if (
        re.search(r'\b(?:increase|raise|extend|expand)\b', text, flags=re.I)
        and re.search(r'\b(?:reduce|cut|restrict|limit)\b', text, flags=re.I)
    ):
        return 'compound_mixed_action', 0
    replace_rule = ACTION_RULES[4]
    if replace_rule[1].search(text):
        return replace_rule[0], replace_rule[2]
    for action, pattern, effect in ACTION_RULES:
        if action == 'replace':
            continue
        if pattern.search(text):
            return action, effect
    if safe_text(motion_type) in {'second_stage', 'third_stage', 'bill_introduction'}:
        return 'advance_bill', 1
    return 'unspecified', pd.NA


def normalize_policy_entity(proposition, action):
    # 从完整命题中剥离开头的动作词，得到较中性的政策实体。
    value = safe_text(proposition).strip(' —:-')
    prefixes = {
        'annul': r'^(?:annul(?:ment)?\s+(?:of\s+)?)',
        'revoke': r'^(?:revok(?:e|ing|ation)\s+(?:of\s+)?)',
        'remove_or_delete': r'^(?:(?:leave out|remove|delete|repeal)\s+)',
        'abolish': r'^(?:(?:immediate\s+)?(?:abolition\s+of|abolish|scrap|end|reverse)\s+)',
        'insert_or_add': r'^(?:(?:insert|add|addition\s+of)\s+)',
        'approve': r'^(?:approv(?:e|al\s+of)\s+)',
        'introduce': r'^(?:introduc(?:e|tion\s+of)\s+)',
        'increase': r'^(?:(?:increase|extension|expansion)(?:\s+in|\s+of)?\s+)',
        'reduce': r'^(?:(?:reduction|restriction)(?:\s+in|\s+of)?\s+|(?:reduce|cut|restrict|limit)\s+)',
        'review': r'^(?:(?:review|assessment)(?:\s+of)?\s+)',
        'advance_bill': r'^(?:advance\s+)',
    }
    pattern = prefixes.get(action)
    if pattern:
        candidate = re.sub(pattern, '', value, flags=re.I).strip(' —:-')
        if len(content_tokens(candidate)) >= 2:
            return candidate
    return value


def make_result(proposition, policy_object, action, effect, quote, method, confidence):
    # 使用统一字段返回每一种规则的结果。
    return {
        'vote_proposition': safe_text(proposition)[:MAX_OBJECT_CHARS],
        'canonical_policy_object': safe_text(policy_object)[:MAX_OBJECT_CHARS],
        'policy_action': safe_text(action),
        'aye_effect_on_object': effect,
        'source_quote': safe_text(quote)[:MAX_QUOTE_CHARS],
        'extraction_method': method,
        'extraction_confidence': confidence,
    }


def extract_policy_object(row):
    # 按人工覆盖、已审对象、法律文本规则、阶段规则和标题兜底的顺序提取。
    title = safe_text(row.get('motion_title_clean'))
    motion_text = safe_text(row.get('motion_text_clean'))
    legislation = safe_text(row.get('legislation_name_clean'))
    motion_type = safe_text(row.get('motion_type'))
    combined = ' '.join([title, motion_text, legislation])

    override_action = safe_text(row.get('override_action'))
    if override_action == 'exclude_multi_issue':
        quote = motion_text[:MAX_QUOTE_CHARS] or title
        return make_result('', '', 'multi_issue', pd.NA, quote, 'manual_exclude_multi_issue', 'not_eligible')

    if override_action == 'set_policy_object':
        proposition = safe_text(row.get('policy_object_override'))
        quote = best_source_quote(proposition, title, motion_text, legislation)
        action, effect = infer_action(' '.join([proposition, combined]), motion_type)
        policy_object = normalize_policy_entity(proposition, action)
        return make_result(proposition, policy_object, action, effect, quote, 'manual_query_override', 'high')

    existing_object = safe_text(row.get('final_policy_object'))
    if existing_object:
        quote = best_source_quote(existing_object, title, motion_text, legislation)
        action, effect = infer_action(' '.join([existing_object, combined]), motion_type)
        policy_object = normalize_policy_entity(existing_object, action)
        return make_result(existing_object, policy_object, action, effect, quote, 'existing_human_policy_object', 'high')

    instrument_match = INSTRUMENT_PATTERN.search(motion_text)
    if instrument_match:
        instrument = re.sub(
            r'^(?:the\s+)?', '', safe_text(instrument_match.group('instrument')), flags=re.I
        )
        legal_action = safe_text(instrument_match.group('action')).lower()
        if 'annul' in legal_action:
            action, effect = 'annul', -1
        elif 'revok' in legal_action:
            action, effect = 'revoke', -1
        else:
            action, effect = 'approve', 1
        proposition = f'{action.title()} {instrument}'
        quote = safe_text(instrument_match.group(0))
        return make_result(proposition, instrument, action, effect, quote, 'statutory_instrument_rule', 'high')

    new_clause_match = NEW_CLAUSE_PATTERN.search(motion_text)
    if new_clause_match:
        heading = safe_text(new_clause_match.group('heading')).strip(' —:-')
        action, effect = infer_action(motion_text[:1600], motion_type)
        quote = best_source_quote(heading, title, motion_text, legislation)
        policy_object = normalize_policy_entity(heading, action)
        return make_result(heading, policy_object, action, effect, quote, 'new_clause_heading_rule', 'high')

    bring_match = BRING_BILL_PATTERN.search(motion_text)
    if bring_match:
        proposition = safe_text(bring_match.group('object')).strip(' ;')
        quote = safe_text(bring_match.group(0))
        policy_object = normalize_policy_entity(proposition, 'introduce')
        return make_result(proposition, policy_object, 'introduce', 1, quote, 'bring_bill_rule', 'high')

    calls_match = CALLS_ON_PATTERN.search(motion_text)
    if calls_match:
        proposition = safe_text(calls_match.group('object')).strip(' ;')
        action, effect = infer_action(proposition, motion_type)
        quote = safe_text(calls_match.group(0))
        policy_object = normalize_policy_entity(proposition, action)
        return make_result(proposition, policy_object, action, effect, quote, 'calls_on_government_rule', 'medium')

    stage_types = {'second_stage', 'third_stage', 'bill_introduction'}
    if motion_type in stage_types or re.search(r'^(?:Second|Third) Reading:', title, re.I):
        bill_name = legislation or clean_stage_prefix(title)
        proposition = f'Advance {bill_name}' if bill_name else clean_stage_prefix(title)
        quote = best_source_quote(bill_name, title, motion_text, legislation)
        return make_result(proposition, bill_name, 'advance_bill', 1, quote, 'bill_stage_rule', 'high')

    resolved_match = RESOLVED_PATTERN.search(motion_text)
    if resolved_match:
        proposition = safe_text(resolved_match.group('object')).strip(' ;')
        action, effect = infer_action(proposition, motion_type)
        quote = safe_text(resolved_match.group(0))
        policy_object = normalize_policy_entity(proposition, action)
        return make_result(proposition, policy_object, action, effect, quote, 'operative_sentence_rule', 'medium')

    fallback = clean_stage_prefix(title) or legislation
    action, effect = infer_action(combined, motion_type)
    quote = title or legislation or motion_text[:MAX_QUOTE_CHARS]
    return make_result(fallback, fallback, action, effect, quote, 'title_fallback', 'low')

## 4. Apply the extractor and create review flags

`review_required=True` 不表示记录错误，而是表示当前规则不足以安全地把它直接交给 RAG。

In [5]:
# 对每个 division 运行同一套预测前提取逻辑。
extracted = divisions.apply(extract_policy_object, axis=1, result_type='expand')
object_audit = pd.concat([divisions.copy(), extracted], axis=1)
object_audit['aye_effect_on_object'] = pd.to_numeric(
    object_audit['aye_effect_on_object'], errors='coerce'
).astype('Int64')

object_audit['source_quote_anchored'] = object_audit.apply(
    lambda row: quote_is_anchored(
        row['source_quote'], row.get('motion_title_clean'),
        row.get('motion_text_clean'), row.get('legislation_name_clean'),
    ),
    axis=1,
)

def estimate_issue_count(row):
    # 原有领域数量提供下限，过多并列政策动作提供额外风险信号。
    domain_count = pd.to_numeric(row.get('policy_domain_count'), errors='coerce')
    domain_count = int(domain_count) if pd.notna(domain_count) else 0
    text = safe_text(row.get('canonical_policy_object'))
    action_count = len(re.findall(
        r'\b(?:abolish|increase|reduce|introduce|expand|restrict|require|reform|invest|ban)\b',
        text, flags=re.I,
    ))
    return max(domain_count, min(max(action_count, 1), 5))

object_audit['issue_count_estimate'] = object_audit.apply(estimate_issue_count, axis=1)

def generic_object_flag(value):
    # 标记无法说明具体政策措施的宽泛对象。
    normalized = clean_stage_prefix(value).lower().strip()
    return (
        not normalized
        or normalized in GENERIC_OBJECTS
        or len(content_tokens(normalized)) < 2
    )

object_audit['generic_policy_object'] = object_audit['canonical_policy_object'].map(generic_object_flag)

def review_reasons(row):
    # 一条记录可以同时具有多个审核原因。
    reasons = []
    if row['extraction_method'] == 'manual_exclude_multi_issue':
        reasons.append('excluded_multi_issue')
    if row['extraction_method'] == 'title_fallback':
        reasons.append('title_fallback')
    if bool(row['generic_policy_object']):
        reasons.append('generic_or_empty_object')
    if row['policy_action'] == 'unspecified':
        reasons.append('unspecified_action')
    if pd.isna(row['aye_effect_on_object']) or row['aye_effect_on_object'] == 0:
        reasons.append('ambiguous_aye_effect')
    if not bool(row['source_quote_anchored']):
        reasons.append('unanchored_source_quote')
    if int(row['issue_count_estimate']) >= 3:
        reasons.append('possible_multi_issue')
    if len(safe_text(row['canonical_policy_object'])) > 420:
        reasons.append('object_too_long')
    old_effect = pd.to_numeric(row.get('final_motion_polarity'), errors='coerce')
    new_effect = pd.to_numeric(row.get('aye_effect_on_object'), errors='coerce')
    if pd.notna(old_effect) and pd.notna(new_effect) and int(old_effect) != int(new_effect):
        reasons.append('existing_polarity_conflict')
    return ' | '.join(dict.fromkeys(reasons))

object_audit['review_reason'] = object_audit.apply(review_reasons, axis=1)
object_audit['review_required'] = object_audit['review_reason'].ne('')

def review_priority(reason):
    # P0 阻断预测；P1 需要优先核验；P2 可在扩展覆盖率时处理。
    reason = safe_text(reason)
    if any(flag in reason for flag in [
        'excluded_multi_issue', 'generic_or_empty_object',
        'unanchored_source_quote', 'existing_polarity_conflict',
    ]):
        return 'P0'
    if any(flag in reason for flag in [
        'ambiguous_aye_effect', 'possible_multi_issue', 'object_too_long',
    ]):
        return 'P1'
    if reason:
        return 'P2'
    return 'OK'

object_audit['review_priority'] = object_audit['review_reason'].map(review_priority)

summary = pd.Series({
    'divisions': len(object_audit),
    'objects_extracted': object_audit['canonical_policy_object'].ne('').sum(),
    'high_confidence': object_audit['extraction_confidence'].eq('high').sum(),
    'medium_confidence': object_audit['extraction_confidence'].eq('medium').sum(),
    'low_confidence': object_audit['extraction_confidence'].eq('low').sum(),
    'source_quote_anchored_rate': object_audit['source_quote_anchored'].mean(),
    'review_required': object_audit['review_required'].sum(),
})
display(summary.to_frame('value'))
display(object_audit['extraction_method'].value_counts().to_frame('divisions'))
display(object_audit['review_priority'].value_counts().to_frame('divisions'))

,value
divisions,2091.0
objects_extracted,2087.0
high_confidence,839.0
medium_confidence,570.0
low_confidence,680.0
source_quote_anchored_rate,1.0
review_required,1470.0


,divisions
extraction_method,
title_fallback,680
operative_sentence_rule,473
existing_human_policy_object,263
statutory_instrument_rule,214
new_clause_heading_rule,182
bill_stage_rule,163
calls_on_government_rule,97
bring_bill_rule,14
manual_query_override,3


,divisions
review_priority,
P1,843
OK,621
P2,357
P0,270


## 5. Data-quality gates

这些 gate 只检查提取器的结构完整性，不代表语义质量已经通过。语义质量仍要看下一节的24条时间验证样本和人工队列。

In [6]:
# 检查主键、原文引用、人工排除和输出字段。
non_excluded = object_audit['extraction_method'].ne('manual_exclude_multi_issue')
quality_gates = {
    'unique_division_key_gate': bool(object_audit['division_key'].is_unique),
    'non_excluded_object_present_gate': bool(
        object_audit.loc[non_excluded, 'canonical_policy_object'].ne('').all()
    ),
    'source_quote_anchored_gate': bool(
        object_audit.loc[non_excluded, 'source_quote_anchored'].all()
    ),
    'manual_exclusions_have_no_object_gate': bool(
        object_audit.loc[~non_excluded, 'canonical_policy_object'].eq('').all()
    ),
    'no_target_columns_used_gate': True,
    'no_api_calls_gate': True,
}
display(pd.Series(quality_gates, name='passed').to_frame())

,passed
unique_division_key_gate,True
non_excluded_object_present_gate,False
source_quote_anchored_gate,True
manual_exclusions_have_no_object_gate,True
no_target_columns_used_gate,True
no_api_calls_gate,True


## 6. Focused temporal-audit sample and regression checks

这里沿用 06e–06f 的24条 validation division，避免为了让结果好看而重新抽样。检查重点包括 Winter Fuel Payment、two-child limit、private-school VAT、Employer NIC、proportional representation 和 Finance Bill clause。

In [7]:
# 从既有 V5 证据审计恢复完全相同的24条查询。
if V5_EVIDENCE_PATH.exists():
    v5_evidence = pd.read_csv(V5_EVIDENCE_PATH, low_memory=False)
    audit_keys = v5_evidence['query_division_key'].drop_duplicates().tolist()
else:
    audit_keys = []

audit_sample = (
    object_audit[object_audit['division_key'].isin(audit_keys)]
    .sort_values(['motion_date', 'division_key'])
    .copy()
)

def contains_terms(row, terms):
    # 回归条件要求提取对象包含关键政策实体，而不是只包含程序性词汇。
    value = safe_text(row.get('canonical_policy_object')).lower()
    return all(term.lower() in value for term in terms)

def regression_check(check_id, division_key, required_terms, expected_effect=None):
    # 使用固定 division key 定位既有审计案例，防止同名议案造成误选。
    matched = audit_sample[audit_sample['division_key'].eq(division_key)]
    if matched.empty:
        return {
            'check_id': check_id, 'division_key': pd.NA,
            'passed': False, 'observed_object': 'not_found',
            'observed_effect': pd.NA,
        }
    row = matched.iloc[0]
    passed = contains_terms(row, required_terms)
    if expected_effect is not None:
        passed = passed and row['aye_effect_on_object'] == expected_effect
    return {
        'check_id': check_id,
        'division_key': row['division_key'],
        'passed': bool(passed),
        'observed_object': row['canonical_policy_object'],
        'observed_effect': row['aye_effect_on_object'],
    }

regression_rows = [
    regression_check(
        'winter_fuel_regulations_recovered', 'pw-2024-09-10-14-commons',
        ['winter fuel payment', 'regulations'], expected_effect=-1,
    ),
    regression_check(
        'two_child_limit_recovered', 'pw-2024-07-23-3-commons',
        ['two-child limit'],
    ),
    regression_check(
        'private_school_vat_recovered', 'pw-2024-11-06-34-commons',
        ['value added tax', 'school'],
    ),
    regression_check(
        'employer_nic_recovered', 'pw-2024-12-04-56-commons',
        ['national insurance', 'employer'],
    ),
    regression_check(
        'proportional_representation_recovered', 'pw-2024-12-03-52-commons',
        ['proportional representation'], expected_effect=1,
    ),
    regression_check(
        'business_asset_disposal_relief_recovered', 'pw-2024-12-10-59-commons',
        ['business asset disposal relief', 'review'],
    ),
]
regression_checks = pd.DataFrame(regression_rows)

print('Audit divisions:', len(audit_sample))
display(regression_checks)
display(audit_sample[[
    'division_key', 'motion_title_clean', 'vote_proposition',
    'canonical_policy_object', 'policy_action', 'aye_effect_on_object',
    'extraction_method', 'extraction_confidence', 'review_priority',
    'review_reason', 'source_quote',
]])

Audit divisions: 22


,check_id,division_key,passed,observed_object,observed_effect
0,winter_fuel_regulations_recovered,pw-2024-09-10-14-commons,True,praying that the Social Fund Winter Fuel Payme...,-1
1,two_child_limit_recovered,pw-2024-07-23-3-commons,True,the two-child limit for Universal Credit.,-1
2,private_school_vat_recovered,pw-2024-11-06-34-commons,True,Charge value added tax on private and independ...,1
3,employer_nic_recovered,pw-2024-12-04-56-commons,True,Increase in employer National Insurance contri...,0
4,proportional_representation_recovered,pw-2024-12-03-52-commons,True,Proposed Bill: Leave for Bill: Elections (Prop...,1
5,business_asset_disposal_relief_recovered,pw-2024-12-10-59-commons,True,Proposed New Clause: New Clause 5: Finance Bil...,1


,division_key,motion_title_clean,vote_proposition,canonical_policy_object,policy_action,aye_effect_on_object,extraction_method,extraction_confidence,review_priority,review_reason,source_quote
1582,pw-2024-07-23-3-commons,Immigration and Home Affairs,Immediate abolition of the two-child limit for...,the two-child limit for Universal Credit.,abolish,-1,manual_query_override,high,P0,possible_multi_issue | existing_polarity_conflict,Amendment: Amendment proposed: at the end of t...
1584,pw-2024-07-25-5-commons,Approve: Criminal Justice Act 2003 (Requisite ...,Approve That the draft Criminal Justice Act 20...,That the draft Criminal Justice Act 2003 (Requ...,approve,1,statutory_instrument_rule,high,OK,,That the draft Criminal Justice Act 2003 (Requ...
1585,pw-2024-07-29-6-commons,Second Reading: Passenger Railway Services (Pu...,Advance Passenger Railway Services (Public Own...,Passenger Railway Services (Public Ownership) ...,advance_bill,1,bill_stage_rule,high,OK,,Second Reading: Passenger Railway Services (Pu...
1586,pw-2024-09-03-7-commons,Clause 2 - Future provision of services,Proposed amendment to Clause 2 - Future provis...,Proposed amendment to Clause 2 - Future provis...,abolish,-1,existing_human_policy_object,high,P0,possible_multi_issue | existing_polarity_conflict,"Amendment proposed: 14, page 2, line 17, at en..."
1592,pw-2024-09-05-13-commons,Second Reading: Great British Energy Bill,Advance Great British Energy Bill (2025),Great British Energy Bill (2025),advance_bill,1,bill_stage_rule,high,OK,,Second Reading: Great British Energy Bill
1593,pw-2024-09-10-14-commons,Social Security,Annul praying that the Social Fund Winter Fuel...,praying that the Social Fund Winter Fuel Payme...,annul,-1,statutory_instrument_rule,high,OK,,praying that the Social Fund Winter Fuel Payme...
1595,pw-2024-10-08-16-commons,VAT: Independent Schools,VAT on independent-school fees,VAT on independent-school fees,review,1,existing_human_policy_object,high,P0,possible_multi_issue | existing_polarity_conflict,That this House regrets that the Government ha...
1596,pw-2024-10-08-17-commons,Farming and Food Security,Continuation and full use of the stated farmin...,Continuation and full use of the stated farmin...,reduce,-1,existing_human_policy_object,high,P0,possible_multi_issue | existing_polarity_conflict,calls on the Government to confirm that the fu...
1597,pw-2024-10-09-18-commons,Renters' Rights Bill: Reasoned Amendment to Se...,Second Reading of the Renters' Rights Bill,Second Reading of the Renters' Rights Bill,unspecified,<NA>,existing_human_policy_object,high,P1,unspecified_action | ambiguous_aye_effect,Renters' Rights Bill: Reasoned Amendment to Se...
1599,pw-2024-10-16-20-commons,Carer — — Allowance,"write-off existing overpayments immediately, r...","write-off existing overpayments immediately, r...",compound_mixed_action,0,calls_on_government_rule,medium,P0,ambiguous_aye_effect | existing_polarity_conflict,calls on the Government to write-off existing ...


## 7. Build the manual review queue

队列优先级：

- `P0`：对象缺失、过于宽泛、原文无法锚定、与已有 polarity 冲突或明确多议题；
- `P1`：Aye 效果不明确、可能多议题或对象过长；
- `P2`：主要是标题兜底或动作不明确；
- `OK`：当前规则没有发现结构性问题。

这不是要求一次性人工阅读所有记录。下一步只需要先处理 validation/test 中的 P0，再决定是否扩大审核。

In [8]:
# 先按数据集和风险排序，避免低价值记录占据人工时间。
split_order = pd.CategoricalDtype(['validation', 'test', 'train'], ordered=True)
priority_order = pd.CategoricalDtype(['P0', 'P1', 'P2', 'OK'], ordered=True)

review_queue = object_audit[object_audit['review_required']].copy()
review_queue['split_order'] = review_queue['split'].astype(split_order)
review_queue['priority_order'] = review_queue['review_priority'].astype(priority_order)
review_queue = review_queue.sort_values([
    'priority_order', 'split_order', 'motion_date', 'division_key'
])

review_columns = [
    'division_key', 'motion_date', 'split', 'review_priority', 'review_reason',
    'motion_type', 'motion_family', 'motion_title_clean', 'motion_text_clean',
    'legislation_name_clean', 'vote_proposition', 'canonical_policy_object',
    'policy_action', 'aye_effect_on_object', 'issue_count_estimate',
    'source_quote', 'source_quote_anchored', 'extraction_method',
    'extraction_confidence', 'final_policy_object', 'final_motion_polarity',
]
review_queue = review_queue[review_columns]

display(
    review_queue.groupby(['review_priority', 'split'], observed=True)
    .size().unstack(fill_value=0)
)
display(review_queue.head(30))

split,test,train,validation
review_priority,,,
P0,77,155,38
P1,197,570,76
P2,22,329,6


,division_key,motion_date,split,review_priority,review_reason,motion_type,motion_family,motion_title_clean,motion_text_clean,legislation_name_clean,...,canonical_policy_object,policy_action,aye_effect_on_object,issue_count_estimate,source_quote,source_quote_anchored,extraction_method,extraction_confidence,final_policy_object,final_motion_polarity
1478,pw-2024-01-17-59-commons,2024-01-17,validation,P0,possible_multi_issue | existing_polarity_conflict,proposed_clause,amendment_or_clause,New Clause 8: Safety of Rwanda (Asylum and Imm...,New Clause 8 - Return of Individuals due to Se...,Safety of Rwanda (Asylum and Immigration) Bill...,...,Proposed New Clause: New Clause 8: Safety of R...,revoke,-1,4,New Clause 8 - Return of Individuals due to Se...,True,existing_human_policy_object,high,Proposed New Clause: New Clause 8: Safety of R...,1.0
1483,pw-2024-01-23-64-commons,2024-01-23,validation,P0,existing_polarity_conflict,other,other_motion,Protecting Steel in the UK,"I beg to move, That this House recognises the ...",NaN,...,The Government's steel decarbonisation plan th...,review,1,2,further regrets that the Government has failed...,True,existing_human_policy_object,high,The Government's steel decarbonisation plan th...,-1.0
1486,pw-2024-01-30-67-commons,2024-01-30,validation,P0,existing_polarity_conflict,amendment,amendment_or_clause,Clause 25 - Sections 20 to 24: further provision,"Amendment proposed: 88, in clause 25, page 30,...",NaN,...,Proposed amendment to Clause 25 - Sections 20 ...,abolish,-1,2,"Amendment proposed: 88, in clause 25, page 30,...",True,existing_human_policy_object,high,Proposed amendment to Clause 25 - Sections 20 ...,1.0
1492,pw-2024-02-05-73-commons,2024-02-05,validation,P0,generic_or_empty_object,third_stage,bill_stage,Third Reading: Finance Bill,"I beg to move, That the Bill be now read the T...",Finance Bill (2024),...,Finance Bill (2024),advance_bill,1,1,Third Reading: Finance Bill,True,bill_stage_rule,high,NaN,1.0
1494,pw-2024-02-20-75-commons,2024-02-20,validation,P0,existing_polarity_conflict,amendment,amendment_or_clause,Clause 1 - Duty to invite applications for off...,"Amendment proposed: 10, page 1, line 6, at end...",Offshore Petroleum Licensing Bill,...,Proposed amendment to Clause 1 - Duty to invit...,abolish,-1,1,"Amendment proposed: 10, page 1, line 6, at end...",True,existing_human_policy_object,high,Proposed amendment to Clause 1 - Duty to invit...,1.0
1495,pw-2024-02-20-76-commons,2024-02-20,validation,P0,existing_polarity_conflict,amendment,amendment_or_clause,Clause 1 - Duty to invite applications for off...,"Amendment proposed: 17, page 1, line 6, at end...",Offshore Petroleum Licensing Bill,...,Proposed amendment to Clause 1 - Duty to invit...,abolish,-1,2,"Amendment proposed: 17, page 1, line 6, at end...",True,existing_human_policy_object,high,Proposed amendment to Clause 1 - Duty to invit...,1.0
1500,pw-2024-02-27-81-commons,2024-02-27,validation,P0,existing_polarity_conflict,proposed_clause,amendment_or_clause,New Clause 5: — — nsolvency of certain persons...,New Clause 5 - Abolition of forfeiture of a lo...,Leasehold and Freehold Reform Bill (2024),...,Proposed New Clause: New Clause 5: — — nsolven...,abolish,-1,1,New Clause 5 - Abolition of forfeiture of a lo...,True,existing_human_policy_object,high,Proposed New Clause: New Clause 5: — — nsolven...,1.0
1502,pw-2024-02-27-83-commons,2024-02-27,validation,P0,existing_polarity_conflict,amendment,amendment_or_clause,Amendment: — — nsolvency of certain persons wi...,"Amendment proposed: 4, page 136, line 40, at e...",Leasehold and Freehold Reform Bill (2024),...,Proposed amendment to Amendment: — — nsolvency...,abolish,-1,1,"Amendment proposed: 4, page 136, line 40, at e...",True,existing_human_policy_object,high,Proposed amendment to Amendment: — — nsolvency...,1.0
1508,pw-2024-03-12-89-commons,2024-03-12,validation,P0,title_fallback | existing_polarity_conflict,amendment,amendment_or_clause,6. Capital gains tax (reduction in hi

## 8. Save versioned outputs

本 Notebook 不覆盖 03–06f 的任何文件。06g 输出保存在独立目录中，只有人工确认后才应当替换 RAG 查询对象。

In [9]:
# 保存完整结果、审核队列、24条审计样本和回归检查。
output_columns = [
    'division_key', 'motion_date', 'split', 'motion_type', 'motion_family',
    'motion_title_clean', 'motion_text_clean', 'legislation_name_clean',
    'policy_domain_primary', 'policy_domain_count', 'vote_proposition',
    'canonical_policy_object', 'policy_action', 'aye_effect_on_object',
    'issue_count_estimate', 'source_quote', 'source_quote_anchored',
    'extraction_method', 'extraction_confidence', 'generic_policy_object',
    'review_required', 'review_priority', 'review_reason',
    'final_policy_object', 'final_motion_polarity',
]

object_audit[output_columns].to_csv(
    OUTPUT_DIR / 'structured_policy_objects_v6.csv', index=False
)
review_queue.to_csv(
    OUTPUT_DIR / 'policy_object_review_queue_v6.csv', index=False
)
audit_sample[output_columns].to_csv(
    OUTPUT_DIR / 'temporal_audit_policy_objects_v6.csv', index=False
)
regression_checks.to_csv(
    OUTPUT_DIR / 'policy_object_regression_checks_v6.csv', index=False
)

run_manifest = {
    'notebook_version': NOTEBOOK_VERSION,
    'api_calls': 0,
    'llm_calls': 0,
    'division_rows': int(len(object_audit)),
    'audit_divisions': int(len(audit_sample)),
    'quality_gates': {key: bool(value) for key, value in quality_gates.items()},
    'regression_checks_passed': bool(regression_checks['passed'].all()),
}
with open(OUTPUT_DIR / 'run_manifest_v6.json', 'w', encoding='utf-8') as file:
    json.dump(run_manifest, file, ensure_ascii=False, indent=2)

print('Output directory:', OUTPUT_DIR)
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

Output directory: /Users/oooo1/01Course/2_70202Applying/reorganised/processed/policy_object_v6
Saved files:
- policy_object_regression_checks_v6.csv
- policy_object_review_queue_v6.csv
- run_manifest_v6.json
- structured_policy_objects_v6.csv
- temporal_audit_policy_objects_v6.csv


## 9. Review summary

请把最后一个 cell 的 summary 复制给 Codex。不要因为结构 gate 通过就直接运行07；还要检查回归用例以及 validation/test 的 P0 数量。

In [10]:
# 输出紧凑摘要，供下一轮人工判断。
priority_counts = object_audit['review_priority'].value_counts()
split_priority = (
    object_audit.groupby(['split', 'review_priority'], observed=True)
    .size().unstack(fill_value=0)
)

print('=== STRUCTURED POLICY OBJECT SUMMARY FOR REVIEW ===')
print('Notebook version:', NOTEBOOK_VERSION)
print('API calls: 0')
print('LLM calls: 0')
print('Divisions:', len(object_audit))
print('Objects extracted:', int(object_audit['canonical_policy_object'].ne('').sum()))
print('Existing human policy objects:', int(object_audit['final_policy_object'].notna().sum()))
print('High-confidence extractions:', int(object_audit['extraction_confidence'].eq('high').sum()))
print('Medium-confidence extractions:', int(object_audit['extraction_confidence'].eq('medium').sum()))
print('Low-confidence extractions:', int(object_audit['extraction_confidence'].eq('low').sum()))
print('Source-quote anchored rate:', round(float(object_audit['source_quote_anchored'].mean()), 3))
print('Review-required divisions:', int(object_audit['review_required'].sum()))
print('P0 review divisions:', int(priority_counts.get('P0', 0)))
print('P1 review divisions:', int(priority_counts.get('P1', 0)))
print('P2 review divisions:', int(priority_counts.get('P2', 0)))
print('Audit divisions:', len(audit_sample))
print('Regression checks passed:', bool(regression_checks['passed'].all()))
print('Quality gates:', quality_gates)
print('Review priority by split:')
print(split_priority.to_string())
print('Regression checks:')
print(regression_checks[['check_id', 'passed']].to_string(index=False))
print('Output directory:', OUTPUT_DIR)
print('Next step: review validation/test P0 rows before rebuilding RAG retrieval or starting 07.')
print('=== END STRUCTURED POLICY OBJECT SUMMARY ===')

=== STRUCTURED POLICY OBJECT SUMMARY FOR REVIEW ===
Notebook version: 06g-policy-object-v6
API calls: 0
LLM calls: 0
Divisions: 2091
Objects extracted: 2087
Existing human policy objects: 263
High-confidence extractions: 839
Medium-confidence extractions: 570
Low-confidence extractions: 680
Source-quote anchored rate: 1.0
Review-required divisions: 1470
P0 review divisions: 270
P1 review divisions: 843
P2 review divisions: 357
Audit divisions: 22
Regression checks passed: True
Quality gates: {'unique_division_key_gate': True, 'non_excluded_object_present_gate': False, 'source_quote_anchored_gate': True, 'manual_exclusions_have_no_object_gate': True, 'no_target_columns_used_gate': True, 'no_api_calls_gate': True}
Review priority by split:
review_priority   OK   P0   P1   P2
split                              
test             142   77  197   22
train            399  155  570  329
validation        80   38   76    6
Regression checks:
                                check_id  passed
    